In [ ]:
import os
import datasets
from datasets import load_dataset

dataset_path = 'datasets'
dataset_name = 'hazemessam/otto-recsys'

cache_dir = 'cache'

os.makedirs(dataset_path, exist_ok=True)
os.makedirs(cache_dir, exist_ok=True)

dataset = load_dataset(dataset_name, cache_dir=cache_dir)

In [ ]:
dataset['train'].shape[0]

In [ ]:
import torch
from tqdm import tqdm
from pathlib import Path
from joblib import Parallel, delayed
from torch.nn import functional as F
from torch.utils.data import Dataset

class RecDataset(Dataset):
    def __init__(
            self, 
            hf_dataset: datasets.dataset_dict.DatasetDict, 
            window_len: int,
            cache_dir: str | None = None
        ):
        self.pad_id = 0
        self.mask_id = 1
        self.unk_id = 2
        self.window_len = window_len

        self.data = self.load_or_process(hf_dataset, cache_dir)
            
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

    def load_or_process(self, dataset, cache_dir):
        if cache_dir is None:
            return self.sample_dataset(dataset)

        path = Path(cache_dir) / f"windows_{self.window_len}.pt"

        if path.exists():
            print(f"Loading cache: {path}")
            return torch.load(path, weights_only=True)

        data = self.sample_dataset(dataset)

        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(data, path)

        return data

    def sampling_session(self, session: dict):
        ids = []

        for event in session['events']:
            if event['type'] != 'clicks':
                continue

            ids.append(event['aid'] + 3)

        ids = torch.tensor(ids)

        dim = 0
        step = 1
        
        if ids.shape[dim] < self.window_len:
            return F.pad(ids, (0, self.window_len - ids.shape[dim]), 'constant', self.pad_id).unfold(dim, self.window_len, step)
        else:
            return ids.unfold(dim, self.window_len, step) 

    def sample_dataset(self, dataset):
        samples = []

        p = Parallel(n_jobs=-1, batch_size="auto")
        samples = p(delayed(self.sampling_session)(session) for session in tqdm(dataset))

        return torch.concat(samples, dim=0)

In [ ]:
window_len = 5

train_ds_path = 'cache/train/'
test_ds_path = 'cache/test/'

train_ds = RecDataset(dataset['train'], window_len, train_ds_path)
test_ds = RecDataset(dataset['test'], window_len, test_ds_path)

In [ ]:
from torch.utils.data import random_split, DataLoader

val_size = 0.1

train, val = random_split(train_ds, [1-val_size, val_size])
test = test_ds

train_loader = DataLoader(
    train,
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    val,
    batch_size=64,
    shuffle=False
)

In [ ]:
def mask_batch(batch):
    mask = (batch != 0).sum(dim=1)
    mask_positions = (torch.rand(mask.shape) * mask).long()

    batch_size = batch.shape[0]
    
    targets = batch[torch.arange(0, batch_size), mask_positions]

    input_ids = batch.clone()   
    input_ids[torch.arange(0, batch_size), mask_positions] = 1

    return input_ids, targets, mask_positions

In [ ]:
import torch.nn as nn

torch.manual_seed(42)

class MultiheadAttention(nn.Module):
    def __init__(
        self,
        num_heads: int,
        out_onehead_dim: int,
        embedding_dim: int,
        window_len: int
    ):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = out_onehead_dim
        self.embedding_dim = embedding_dim
        self.window_len = window_len

        # Матрицы проекций Q, K, V
        self.W_q = self.__initialize_weights(
            embedding_dim, num_heads * out_onehead_dim
        )
        self.W_k = self.__initialize_weights(
            embedding_dim, num_heads * out_onehead_dim
        )
        self.W_v = self.__initialize_weights(
            embedding_dim, num_heads * out_onehead_dim
        )
 
        self.W_o = self.__initialize_weights(
            num_heads * out_onehead_dim, embedding_dim
        )

    def __initialize_weights(self, in_dim, out_dim):
        return nn.Parameter(
            nn.init.xavier_uniform_(
                torch.empty(in_dim, out_dim)
            )
        )

    def forward(self, X, padding_mask=None):
        B, W, D = X.shape

        Q = X @ self.W_q
        K = X @ self.W_k
        V = X @ self.W_v

        Q = Q.reshape(B, W, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(B, W, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(B, W, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if padding_mask is not None:
            scores = scores.masked_fill(
                padding_mask[:, None, None, :],
                torch.finfo(scores.dtype).min
            )

        attention = torch.softmax(scores, dim=-1)

        Z = attention @ V

        Z = Z.transpose(1, 2).contiguous()
        Z = Z.reshape(B, W, self.num_heads * self.head_dim)

        return Z @ self.W_o
        

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, head_dim, window_len):
        super().__init__()

        self.attention = MultiheadAttention(num_heads, head_dim, embedding_dim, window_len)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.GELU(),
            nn.Linear(embedding_dim * 4, embedding_dim)
        )

        self.dropout = nn.Dropout(0.1)

    def forward(self, X, padding_mask):
        X = self.norm1(X + self.dropout(self.attention(X, padding_mask)))
        X = self.norm2(X + self.dropout(self.feedforward(X)))

        return X


class BERT4Rec(nn.Module):
    def __init__(
            self,
            window_len,
            vocab_size,
        ):
        super().__init__()

        self.embedding_len = 64
        self.multihead_size = 4
        self.head_dim = 32
        self.num_layers = 2

        self.positional_embeddings = nn.Embedding(window_len, self.embedding_len)
        self.vocabualry_embeddings = nn.Embedding(
            vocab_size, self.embedding_len, padding_idx=0
        )

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(
                self.embedding_len,
                self.multihead_size,
                self.head_dim,
                window_len
            )
            for _ in range(self.num_layers)
        ])

        self.output_norm = nn.LayerNorm(self.embedding_len)

    def forward(self, input_ids, mask_positions):
        B, W = input_ids.shape

        positions = torch.arange(W, device=input_ids.device)

        X = (
            self.vocabualry_embeddings(input_ids)
            + self.positional_embeddings(positions)
        )

        padding_mask = input_ids == 0

        for block in self.transformer_blocks:
            X = block(X, padding_mask)

        X = self.output_norm(X)

        X = X[torch.arange(B, device=X.device), mask_positions]

        logits = X @ self.vocabualry_embeddings.weight.T

        return logits

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def train_model(model, train_loader, val_loader, epochs=3, lr=1e-4):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    val_losses = []
    best_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            inputs, targets, positions = mask_batch(batch)

            inputs = inputs.to(device)
            targets = targets.to(device)
            positions = positions.to(device)

            optimizer.zero_grad()

            logits = model(inputs, positions)
            loss = criterion(logits, targets)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                positions = (batch != 0).sum(dim=1) - 1
                indices = torch.arange(batch.shape[0])

                targets = batch[indices, positions]
                inputs = batch.clone()
                inputs[indices, positions] = 1

                logits = model(inputs.to(device), positions.to(device))
                loss = criterion(logits, targets.to(device))

                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch {epoch + 1}: train={train_loss:.4f}, val={val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "bert4rec_best.pt")

    plt.plot(range(1, epochs + 1), train_losses, label="Train")
    plt.plot(range(1, epochs + 1), val_losses, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("CrossEntropyLoss")
    plt.legend()
    plt.grid()
    plt.show()

    return train_losses, val_losses

In [ ]:
vocab_size = max(
    train_ds.data.max().item(),
    test_ds.data.max().item()
) + 1

model = BERT4Rec(window_len=5, vocab_size=vocab_size)

In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, RandomSampler
from tqdm.auto import tqdm


def train_model_in_parts(
        model,
        train_dataset,
        eval_loader,
        part_size=0.05,
        epochs_per_part=4,
        lr=1e-4,
        batch_size=64,
        k=10
    ):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    num_parts = math.ceil(1 / part_size)
    samples_per_part = int(len(train_dataset) * part_size)

    history = []

    for part in range(num_parts):
        print(f"\n{'=' * 40}")
        print(f"Part {part + 1}/{num_parts}")

        sampler = RandomSampler(
            train_dataset,
            replacement=True,
            num_samples=samples_per_part
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler
        )

        for epoch in range(epochs_per_part):
            model.train()
            train_loss = 0

            for batch in tqdm(
                train_loader,
                desc=f"Part {part + 1}, epoch {epoch + 1}/{epochs_per_part}"
            ):
                inputs, targets, positions = mask_batch(batch)

                inputs = inputs.to(device)
                targets = targets.to(device)
                positions = positions.to(device)

                optimizer.zero_grad()

                logits = model(inputs, positions)
                loss = criterion(logits, targets)

                loss.backward()
                optimizer.step()

                train_loss += loss.item()

            train_loss /= len(train_loader)

            print(
                f"Part {part + 1}, "
                f"epoch {epoch + 1}: "
                f"train={train_loss:.4f}"
            )

        # Сохраняется после полного обучения на текущем part.
        # Файл перезаписывается, поэтому место на диске не расходуется.
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "part": part + 1
            },
            "bert4rec_checkpoint.pt"
        )

        print(f"\nПроверка после part {part + 1}")

        recall, ndcg = evaluate_recommendations(
            model,
            eval_loader,
            k=k
        )

        history.append(
            {
                "part": part + 1,
                "train_loss": train_loss,
                f"recall@{k}": recall,
                f"ndcg@{k}": ndcg
            }
        )

    return model, history

In [ ]:
model = train_model_in_parts(
    model=model,
    train_dataset=train,
    val_loader=val_loader,
    part_size=0.05,
    epochs_per_part=4,
    batch_size=64
)

In [ ]:
import math

def evaluate_recommendations(model, data_loader, k=10):
    device = next(model.parameters()).device

    model.eval()

    total = 0
    hits = 0
    ndcg = 0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Recall@{k}"):
            lengths = (batch != 0).sum(dim=1)

            valid_mask = lengths > 0

            batch = batch[valid_mask]
            lengths = lengths[valid_mask]

            if len(batch) == 0:
                continue

            positions = lengths - 1
            indices = torch.arange(batch.shape[0])

            targets = batch[indices, positions]

            inputs = batch.clone()
            inputs[indices, positions] = 1

            logits = model(
                inputs.to(device),
                positions.to(device)
            )

            # Не рекомендуем PAD, MASK и UNK
            logits[:, :3] = float("-inf")

            top_items = torch.topk(logits, k=k, dim=1).indices.cpu()

            matches = top_items == targets.unsqueeze(1)

            batch_hits = matches.any(dim=1)
            hits += batch_hits.sum().item()

            ranks = matches.float().argmax(dim=1) + 1

            ndcg += (
                batch_hits.float()
                / torch.log2(ranks.float() + 1)
            ).sum().item()

            total += len(batch)

    recall = hits / total
    ndcg = ndcg / total

    print(f"Recall@{k}: {recall:.4f}")
    print(f"NDCG@{k}:   {ndcg:.4f}")

    return recall, ndcg

In [ ]:
test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False
)

evaluate_recommendations(model, test_loader, k=10)